# Week 20: MLOps - CI/CD, Monitoring, and LLM Observability

## Where we are

Last week you turned the fraud classifier into a production asset: a SageMaker training job, an MLflow run, a model registered in the Model Registry, a deployed endpoint, and a `classify_with_finetuned_model` tool plugged into `week19_supervisor`. That is the artifact. This week we operate it.

## Learning objectives

By the end of this lab you will be able to:

1. Add OpenTelemetry-based tracing to a Strands agent and view per-decision traces in Langfuse without modifying agent code
2. Verify SageMaker data capture on an endpoint and schedule a Model Monitor job
3. Create a CloudWatch alarm on endpoint latency that notifies an SNS topic
4. Explain when to reach for LiteLLM versus a full agent framework

## The mental model

Week 18 measured RAG quality OFFLINE on a fixed eval set with RAGAS. That tells you what the system can do in a lab. Week 20 measures the SAME system ONLINE on live traffic with Langfuse, Model Monitor, and CloudWatch. That tells you what the system is actually doing right now, in production, on real users.

## Environment Setup

**Platform**: Azure Databricks (plain Runtime 15.4 LTS, Python 3.10 - not the ML runtime).

The next code cell uses `%pip install` (a Databricks magic) to install or upgrade the libraries this notebook needs. The default cluster `boto3` predates the Bedrock Converse API, so we upgrade it here. After the install finishes, `dbutils.library.restartPython()` restarts the Python kernel so the new versions are picked up - once it restarts, re-run from the top.

**Libraries installed**:

- `numpy<2`, `pandas<2` (pinned first to protect the runtime pyarrow)
- `boto3>=1.36` (Bedrock Converse API support)
- `sagemaker>=2.230,<3` (v3 breaks `from sagemaker import get_execution_role`)
- `strands-agents>=1.37,<2`
- `strands-agents-tools>=0.2`
- `opentelemetry-api`, `opentelemetry-sdk`, `opentelemetry-exporter-otlp` (required by Strands' OTLP exporter in Part 1; not pulled transitively)
- `litellm>=1.50` (unified LLM interface, Part 4)
- `langfuse>=2.50,<3` (observability backend, Part 1)

**Already on the cluster** (do NOT pip-install these - reinstalling them breaks the DBR REPL): `pyspark`.

Run the cell below, restart the kernel when prompted, then continue.


In [ ]:
# Install/upgrade the libraries this notebook needs. Safe to re-run.
# Databricks will ask you to restart the kernel afterwards - restartPython does that.
# This is plain DBR 15.4 LTS (not the ML runtime). numpy<2/pandas<2 are pinned
# FIRST to protect the runtime pyarrow so no transitive dependency bumps numpy to
# 2.x and crashes the kernel. The opentelemetry-exporter-otlp package is REQUIRED
# by StrandsTelemetry().setup_otlp_exporter() in Part 1; strands-agents does not
# pull it transitively.
%pip install --quiet "numpy<2" "pandas<2" "boto3>=1.36" "sagemaker>=2.230,<3" "strands-agents>=1.37,<2" "strands-agents-tools>=0.2" "opentelemetry-api" "opentelemetry-sdk" "opentelemetry-exporter-otlp" "litellm>=1.50" "langfuse>=2.50,<3"

dbutils.library.restartPython()

# Verify versions (use importlib.metadata - never pkg.__version__).
from importlib.metadata import version

for pkg in ["boto3", "sagemaker", "strands-agents", "litellm", "langfuse",
            "opentelemetry-exporter-otlp"]:
    try:
        print(f"{pkg:30s} {version(pkg)}")
    except Exception as e:
        print(f"{pkg:30s} NOT INSTALLED ({e})")


In [ ]:
# Standard MLOps setup on Databricks: pull secrets, build boto3 clients.

import os
import json
import base64
import boto3

# Per-student AWS keys live in aws-course-creds-NN; class-wide config is in
# aws-course-shared. Derive this student's scope from the Databricks identity.
_user = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook().getContext().userName().get()
)
_num = _user.split("@")[0].split("-")[1] if _user.startswith("student-") else "01"
creds_scope = f"aws-course-creds-{_num}"

# Long-lived IAM-user keys (no AWS_SESSION_TOKEN).
AWS_ACCESS_KEY_ID     = dbutils.secrets.get(scope=creds_scope, key="aws-access-key-id")
AWS_SECRET_ACCESS_KEY = dbutils.secrets.get(scope=creds_scope, key="aws-secret-access-key")
AWS_REGION            = dbutils.secrets.get(scope="aws-course-shared", key="aws-region")

os.environ["AWS_ACCESS_KEY_ID"] = AWS_ACCESS_KEY_ID
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY
os.environ["AWS_REGION"] = AWS_REGION
os.environ["AWS_DEFAULT_REGION"] = AWS_REGION
# LiteLLM (Part 4) reads AWS_REGION_NAME specifically.
os.environ["AWS_REGION_NAME"] = AWS_REGION

# Clients used across the notebook
sagemaker_client = boto3.client("sagemaker", region_name=AWS_REGION)
sm_runtime = boto3.client("sagemaker-runtime", region_name=AWS_REGION)
cloudwatch = boto3.client("cloudwatch", region_name=AWS_REGION)
s3 = boto3.client("s3", region_name=AWS_REGION)

# This class has 3 cohorts of 22 students. Each cohort gets its OWN endpoint
# so 60 people never contend on one. The endpoint name follows the pattern
# the instructor deployed: fraud-classifier-endpoint-cohort-{1,2,3}.
# Cohort is derived from the student number: 1-22 -> 1, 23-44 -> 2, 45-66 -> 3.
cohort = (int(_num) - 1) // 22 + 1
ENDPOINT_NAME_BASE = dbutils.secrets.get(
    scope="aws-course-shared", key="endpoint-name-base")
ENDPOINT_NAME = f"{ENDPOINT_NAME_BASE}-{cohort}"

# Carried over from Week 19
BEDROCK_MODEL_ID = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"

print("AWS region:", AWS_REGION)
print("Cohort:", cohort)
print("Endpoint:", ENDPOINT_NAME)
print("Bedrock model:", BEDROCK_MODEL_ID)

In [ ]:
# Pre-flight: fail loud now if anything we depend on is missing.

# 1. Endpoint is in service
resp = sagemaker_client.describe_endpoint(EndpointName=ENDPOINT_NAME)
assert resp["EndpointStatus"] == "InService", (
    f"Endpoint {ENDPOINT_NAME} is {resp['EndpointStatus']}. "
    "Ask your instructor to redeploy the Week 19 endpoint."
)
print("Endpoint status:", resp["EndpointStatus"])

# 2. Bedrock LLM responds
bedrock_runtime = boto3.client("bedrock-runtime", region_name=AWS_REGION)
try:
    bedrock_runtime.converse(
        modelId=BEDROCK_MODEL_ID,
        messages=[{"role": "user", "content": [{"text": "ping"}]}],
        inferenceConfig={"maxTokens": 10, "temperature": 0},
    )
    print("Bedrock LLM probe: ok")
except Exception as e:
    print("Ask your instructor to enable Bedrock access for", BEDROCK_MODEL_ID)
    raise

print("Pre-flight checks passed.")

In [ ]:
# Week 19 fraud-investigation supervisor, rebuilt INLINE so this notebook is
# fully self-contained (no %run, no external helper notebook). It is the same
# agent we operated in Week 19: a Strands Agent backed by a Bedrock model,
# wired to a set of fraud-investigation tools. We trace THIS agent in Part 1 -
# we do NOT change it once Langfuse is wired.
#
# The supervisor has three tools:
#   1. classify_with_finetuned_model - calls the Week 19 SageMaker endpoint
#   2. check_fraud_policy            - looks up the Bread Financial fraud rules
#   3. score_transaction_risk        - a small deterministic heuristic check
# The LLM decides which tools to call and in what order; each tool call shows
# up as its own span in Langfuse.

import json
from strands import Agent, tool
from strands.models import BedrockModel


@tool
def classify_with_finetuned_model(narrative: str) -> str:
    """Classify a transaction narrative as fraud or not_fraud.

    Invokes the Week 19 fine-tuned SageMaker endpoint (a HuggingFace
    DistilBERT text classifier). Pass the free-text transaction narrative;
    the endpoint returns the predicted label and a confidence score.

    Args:
        narrative: Free-text description of the transaction.

    Returns:
        The raw JSON string returned by the endpoint, e.g.
        '[{"label": "fraud", "score": 0.93}]'.
    """
    out = sm_runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Body=json.dumps({"inputs": narrative}),
    )
    return out["Body"].read().decode()


@tool
def check_fraud_policy(topic: str) -> str:
    """Look up the Bread Financial fraud-handling policy for a topic.

    Use this to find the documented rule for a situation before recommending
    an action, so the recommendation is grounded in policy and auditable.

    Args:
        topic: A short policy topic such as 'high amount', 'cross border',
            'new customer', 'velocity', or 'card not present'.

    Returns:
        The policy text for that topic, or a default if none matches.
    """
    policy = {
        "high amount": (
            "Transactions over 1000 USD require a step-up review and a "
            "second approver before the funds are released."
        ),
        "cross border": (
            "Cross-border merchant transactions get a +1 risk weight and "
            "must be checked against the sanctioned-country list."
        ),
        "new customer": (
            "Accounts under 30 days old route to manual review regardless "
            "of the classifier score."
        ),
        "velocity": (
            "More than 3 transactions in 10 minutes from one account is a "
            "velocity flag; block pending customer contact."
        ),
        "card not present": (
            "Card-not-present purchases above 250 USD require a one-time "
            "passcode confirmation."
        ),
    }
    return policy.get(
        topic.lower(),
        "No specific policy for that topic; apply the standard risk score.",
    )


@tool
def score_transaction_risk(amount_usd: float, is_cross_border: bool,
                           account_age_days: int) -> str:
    """Compute a quick deterministic risk score for a transaction.

    This is a cheap heuristic the supervisor can use ALONGSIDE the fine-tuned
    classifier - the classifier reads the narrative text, this reads the
    structured fields. Combining a model signal with a rules signal is a
    common fraud-investigation pattern.

    Args:
        amount_usd: Transaction amount in US dollars.
        is_cross_border: True if the merchant is in a different country.
        account_age_days: Age of the customer account in days.

    Returns:
        A human-readable line with the numeric score and a LOW/MEDIUM/HIGH
        risk band.
    """
    score = 0
    if amount_usd > 1000:
        score += 2
    elif amount_usd > 250:
        score += 1
    if is_cross_border:
        score += 1
    if account_age_days < 30:
        score += 2
    band = "LOW" if score <= 1 else "MEDIUM" if score <= 3 else "HIGH"
    return f"risk_score={score} band={band}"


# The Bedrock-backed model. BedrockModel picks up the AWS credentials that
# Cell 1 exported to the environment; region_name keeps it pinned.
supervisor_model = BedrockModel(
    model_id=BEDROCK_MODEL_ID,
    region_name=AWS_REGION,
    temperature=0,
)

# A clear, role-specific system prompt. This is what makes the agent a
# fraud-investigation SUPERVISOR rather than a generic chatbot.
SUPERVISOR_SYSTEM_PROMPT = (
    "You are a fraud investigation supervisor at Bread Financial. "
    "For each transaction you are given, run a thorough investigation:\n"
    "1. Call classify_with_finetuned_model on the transaction narrative to "
    "get the model's fraud label and confidence.\n"
    "2. Call check_fraud_policy for any relevant topic (for example "
    "'high amount', 'cross border', 'new customer', 'velocity', or "
    "'card not present').\n"
    "3. When you know the amount, whether it is cross-border, and the "
    "account age, call score_transaction_risk for a second, rules-based "
    "signal.\n"
    "Then weigh the model signal against the policy and the risk score and "
    "give a final recommendation of APPROVE, REVIEW, or BLOCK, followed by "
    "one concise sentence of justification that cites the evidence you used."
)

# Build the agent. The variable name week19_supervisor is the contract every
# downstream cell depends on - do not rename it.
week19_supervisor = Agent(
    model=supervisor_model,
    tools=[classify_with_finetuned_model, check_fraud_policy, score_transaction_risk],
    system_prompt=SUPERVISOR_SYSTEM_PROMPT,
)

print("Supervisor ready:", type(week19_supervisor).__name__)
# Agent.tool_names is the public Strands API for the registered tool names.
print("Tools:", week19_supervisor.tool_names)


## Part 1 - Langfuse observability (online traces)

### The problem

You ship `week19_supervisor`. A user complains it gave a wrong fraud decision on transaction `txn_009812` at 14:32 UTC yesterday. You need to answer:

- Which tools did the supervisor call, in what order?
- What did `classify_with_finetuned_model` return?
- Which policy did `check_fraud_policy` return?
- How long did the whole decision take, and what did it cost in tokens?

Without tracing this is impossible. RAGAS does not help, because RAGAS is offline. You need a per-request flight recorder. That is Langfuse.

### How Strands + Langfuse fit together

Strands emits OpenTelemetry spans for every model call and tool call. Langfuse speaks OTLP. We point Strands at Langfuse with five lines, then re-run the agent. No code change to the agent itself.

The magic is in `StrandsTelemetry().setup_otlp_exporter()`. After that call, every `agent(prompt)` invocation produces a trace in the Langfuse UI. In the demo and Lab 1 below we go one step further and group each investigation's spans into a Langfuse **Session** keyed on the real `transaction_id`.

In [ ]:
# Five-line wiring. After this cell every supervisor call shows up in Langfuse.

import base64
import os
from strands.telemetry import StrandsTelemetry

LANGFUSE_PUBLIC_KEY = dbutils.secrets.get(scope="aws-course-shared", key="langfuse-public-key")
LANGFUSE_SECRET_KEY = dbutils.secrets.get(scope="aws-course-shared", key="langfuse-secret-key")
LANGFUSE_HOST = dbutils.secrets.get(scope="aws-course-shared", key="langfuse-host")

# Langfuse OTLP endpoint expects basic auth with base64(public:secret)
LANGFUSE_AUTH = base64.b64encode(
    f"{LANGFUSE_PUBLIC_KEY}:{LANGFUSE_SECRET_KEY}".encode()
).decode()

os.environ["OTEL_EXPORTER_OTLP_ENDPOINT"] = LANGFUSE_HOST + "/api/public/otel"
os.environ["OTEL_EXPORTER_OTLP_HEADERS"] = f"Authorization=Basic {LANGFUSE_AUTH}"
# Optional: a service name groups traces in the UI
os.environ["OTEL_SERVICE_NAME"] = "bread-academy-week20"

StrandsTelemetry().setup_otlp_exporter()
print("Langfuse OTLP exporter wired. Host:", LANGFUSE_HOST)

### Where the transactions live

The fraud transactions are real rows in a Delta table:
`bread_academy.course_data.fraud_transactions`. This is the same table Week 19
trained the classifier on. It has about 45,000 rows. Each row has a real
`transaction_id` (format `txn_NNNNNN`), a `customer_id` (`cust_NNNNN`), the
`amount`, `merchant_country`, `merchant_category`, an `is_fraud` label, and a
free-text `narrative` describing the transaction.

We query this table with Spark. The next cell is a DEMO: it pulls one real
fraud transaction and runs it through the supervisor. In Lab 1 you will query
a handful of real transactions yourself and investigate each one.

In [ ]:
# DEMO: query one real fraud transaction from Spark, then investigate it
# as a Langfuse session.

# Pull a single real fraud row from the Delta table.
demo_row = (
    spark.read.table("bread_academy.course_data.fraud_transactions")
    .filter("is_fraud = 1")
    .select("transaction_id", "customer_id", "amount",
            "merchant_country", "narrative")
    .orderBy("transaction_id")
    .limit(1)
    .collect()[0]
)

print("Real transaction pulled from the table:")
print("  id        :", demo_row["transaction_id"])
print("  customer  :", demo_row["customer_id"])
print("  amount    :", demo_row["amount"])
print("  country   :", demo_row["merchant_country"])
print("  narrative :", demo_row["narrative"])
print()

# Build a fresh agent whose trace_attributes carry session.id. Every span
# this agent emits is grouped under one Langfuse session named after the
# real transaction_id. trace_attributes is a CONSTRUCTOR argument - it
# cannot be set on the agent(prompt) call.
demo_agent = Agent(
    model=supervisor_model,
    tools=[classify_with_finetuned_model, check_fraud_policy,
           score_transaction_risk],
    system_prompt=SUPERVISOR_SYSTEM_PROMPT,
    trace_attributes={
        "session.id": demo_row["transaction_id"],
        "user.id": demo_row["customer_id"],
        "tags": ["week20-demo"],
    },
)

# Build the prompt from the REAL narrative text.
demo_prompt = (
    f"Investigate transaction {demo_row['transaction_id']} for customer "
    f"{demo_row['customer_id']}. Transaction narrative: "
    f"\"{demo_row['narrative']}\". Use the fine-tuned classifier on the "
    "narrative, check policy, and recommend an action."
)

response = demo_agent(demo_prompt)
print(str(response)[:500])
print()
print("Open", LANGFUSE_HOST, "-> Sessions. Session", demo_row["transaction_id"],
      "groups:")
print("- the top-level supervisor span")
print("- each tool invocation as a child span")
print("- the bedrock-runtime converse calls with token counts")

### Lab 1 - Investigate five real transactions, one Langfuse session each (15 min)

You are the on-call data scientist. In the demo above you ran ONE real
transaction through the supervisor. Now do five, and make each investigation
its own Langfuse SESSION.

#### Why sessions

A single supervisor call produces a Langfuse TRACE. But one real
investigation can be several calls, and at production scale you want every
trace about transaction `txn_000037` grouped together so you can read the
whole investigation as one unit. That grouping is a Langfuse **Session**.

Langfuse builds a Session from a span attribute called `session.id`. Strands
lets you stamp that attribute on every span an agent emits through the
`trace_attributes` constructor argument:

```python
agent = Agent(
    model=supervisor_model,
    tools=[...],
    system_prompt=SUPERVISOR_SYSTEM_PROMPT,
    trace_attributes={
        "session.id": "txn_000037",   # this trace joins session txn_000037
        "user.id": "cust_01437",
        "tags": ["week20-lab1"],
    },
)
```

Important: `trace_attributes` is set at CONSTRUCTION time - you cannot pass
it to `agent(prompt)`. So to give each transaction its own session, you
build a fresh agent inside the loop, once per transaction.

#### Your task

1. Query five real fraud transactions from
   `bread_academy.course_data.fraud_transactions` (filter `is_fraud = 1`).
   Select `transaction_id`, `customer_id`, `amount`, `merchant_country`,
   and `narrative`.
2. Loop over the five rows. For each one:
   - build a fresh `Agent` with `trace_attributes` whose `session.id` is
     that row's real `transaction_id` and `user.id` is its `customer_id`;
   - reuse `supervisor_model`, the three tools, and
     `SUPERVISOR_SYSTEM_PROMPT` from the supervisor cell;
   - build the prompt from the real `narrative` and invoke the agent.
3. Open Langfuse -> Sessions. You should see five sessions, one per
   `transaction_id`. Open the slowest and answer:
   - Which transaction took the longest end-to-end?
   - Which tool dominated the latency for that transaction?
   - What were the total input and output tokens for the slowest one?

Hints:
- `spark.read.table(...).filter("is_fraud = 1").select(...).limit(5).collect()`
  gives you five `Row` objects.
- The three tool functions (`classify_with_finetuned_model`,
  `check_fraud_policy`, `score_transaction_risk`) are already defined - pass
  them straight into the `tools=[...]` list.
- In the Langfuse UI use the Sessions tab, not the Traces tab.

### Stretch

Add a `langfuse.environment` attribute (for example `"classroom"`) to
`trace_attributes` and confirm it shows up on every span in the session.

### Homework extension

Pipe the production supervisor through Langfuse for 24 hours of real traffic
(simulated by replaying the `fraud_transactions` table), one session per
transaction. Build a Databricks SQL dashboard from the Langfuse exported
events to track p95 latency by tool.

In [ ]:
# Lab 1 SOLUTION.
# Step 1: query five real fraud transactions from the Delta table.
# Step 2: loop. For each row build a FRESH agent whose trace_attributes
#         carry session.id = transaction_id, then invoke it.

from strands import Agent

# Step 1 - five real fraud rows. is_fraud = 1 keeps it to fraud cases;
# select the columns the supervisor's tools actually use.
lab_rows = (
    spark.read.table("bread_academy.course_data.fraud_transactions")
    .filter("is_fraud = 1")
    .select("transaction_id", "customer_id", "amount",
            "merchant_country", "narrative")
    .orderBy("transaction_id")
    .limit(5)
    .collect()
)

# Step 2 - one fresh agent per transaction so each investigation is its own
# Langfuse session. trace_attributes is constructor-only, so the agent must
# be rebuilt inside the loop to vary session.id.
results = []
for row in lab_rows:
    investigation_agent = Agent(
        model=supervisor_model,
        tools=[classify_with_finetuned_model, check_fraud_policy,
               score_transaction_risk],
        system_prompt=SUPERVISOR_SYSTEM_PROMPT,
        trace_attributes={
            "session.id": row["transaction_id"],
            "user.id": row["customer_id"],
            "tags": ["week20-lab1"],
        },
    )
    prompt = (
        f"Investigate transaction {row['transaction_id']} for customer "
        f"{row['customer_id']}. Transaction narrative: "
        f"\"{row['narrative']}\". Use the fine-tuned classifier on the "
        "narrative, check policy, and recommend an action."
    )
    out = investigation_agent(prompt)
    results.append({
        "transaction_id": row["transaction_id"],
        "amount": row["amount"],
        "text": str(out)[:200],
    })

print(f"Ran {len(results)} real transactions, one session each.")
for r in results:
    print(" ", r["transaction_id"], "amount", r["amount"])
print()
print("Done. Open", LANGFUSE_HOST, "-> Sessions to inspect.")

### Think about it

Langfuse stores every prompt and every tool argument by default. A fraud transaction record contains a customer ID and a dollar amount. Is that PII for your jurisdiction? If it is, what would you change about this setup before pointing it at real production traffic? Consider redaction, self-hosted Langfuse, sampling, and retention windows.

## Part 2 - SageMaker data capture and Model Monitor

### What Langfuse will NOT tell you

Langfuse traces the supervisor's decisions. It does not watch the fine-tuned classifier endpoint at the input-feature level. If next month the distribution of `amount` shifts because of a new product launch, the endpoint will keep returning predictions and Langfuse will keep showing happy traces, while the classifier silently goes off the rails.

That is what SageMaker Model Monitor is for. It samples requests and responses into S3 (data capture), then a scheduled job compares the captured distribution to a baseline you computed from training data. If a feature drifts beyond a threshold, the monitoring job writes a violation report.

### Data capture is already on - your instructor enabled it

Data capture is configured on the `EndpointConfig`, not the endpoint directly. There are two enablement points:

1. At endpoint-creation time (cleanest)
2. Updating an existing endpoint's config after the fact

Your cohort endpoint was deployed **with data capture already on**, at creation time, by the instructor setup script. That is deliberate: enabling capture means an `update_endpoint` call, and an endpoint can only run one update at a time. If 60 students each tried to enable capture themselves, all but the first would hit `ValidationException: cannot update in-progress endpoint`. So the instructor does it once, per cohort endpoint.

For reference, this is the code the instructor ran (you do NOT run it - it is shown so you know the API):

```python
from sagemaker.model_monitor import DataCaptureConfig

capture_cfg = DataCaptureConfig(
    enable_capture=True,
    sampling_percentage=100,
    destination_s3_uri=f"s3://{bucket}/fraud-classifier/data-capture/cohort-{cohort}",
    capture_options=["REQUEST", "RESPONSE"],
)
model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.xlarge",
    endpoint_name=f"fraud-classifier-endpoint-cohort-{cohort}",
    data_capture_config=capture_cfg,
)
```

The next cell just VERIFIES capture is on and prints where the captured traffic lands.

In [ ]:
# Verify data capture is enabled on this cohort's endpoint.
# This is READ-ONLY - no update_endpoint call, so it is safe to run any
# number of times and safe for all 60 students to run at once.

bucket = dbutils.secrets.get(scope="aws-course-shared", key="course-s3-bucket")

# The endpoint's active config carries the DataCaptureConfig.
active_cfg_name = sagemaker_client.describe_endpoint(
    EndpointName=ENDPOINT_NAME
)["EndpointConfigName"]
active_cfg = sagemaker_client.describe_endpoint_config(
    EndpointConfigName=active_cfg_name
)

capture = active_cfg.get("DataCaptureConfig")
if not capture or not capture.get("EnableCapture"):
    raise RuntimeError(
        f"Data capture is NOT enabled on {ENDPOINT_NAME}. "
        "Ask your instructor to redeploy the cohort endpoint with "
        "data capture (instructor_setup_aws.py does this at deploy time)."
    )

capture_s3_uri = capture["DestinationS3Uri"]
print("Data capture: ENABLED")
print("Sampling percentage:", capture.get("InitialSamplingPercentage"))
print("Capture S3:", capture_s3_uri)

In [ ]:
# Send a few invocations so there is captured traffic to look at.
# The endpoint is already InService (the instructor deployed it), so there
# is no update to wait for.
import time
import json

# The fraud classifier is a HuggingFace DistilBERT text model - it expects
# {"inputs": "<narrative text>"}.
sample_payload = json.dumps({
    "inputs": "Card-not-present purchase of 482.50 at an online electronics "
              "retailer, 2 hours after the previous transaction."
})
for i in range(10):
    sm_runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Body=sample_payload,
    )
print("Sent 10 invocations to", ENDPOINT_NAME)

# Inspect what landed in S3. Capture can take a minute to flush.
# capture_s3_uri is "s3://<bucket>/<prefix>" - strip the scheme for list.
capture_prefix = capture_s3_uri.replace(f"s3://{bucket}/", "")
time.sleep(60)
listing = s3.list_objects_v2(Bucket=bucket, Prefix=capture_prefix)
found = listing.get("Contents", [])
print(f"Capture objects under {capture_prefix}: {len(found)}")
for obj in found[:5]:
    print(" ", obj["Key"], obj["Size"], "bytes")

### What Model Monitor actually is, and what we are about to build

So far you have ONE of the three pieces Model Monitor needs: data capture is
on, so every request and response the endpoint sees is being written to S3.
That alone monitors nothing. Monitoring is a comparison, and a comparison
needs a baseline and a job that runs the comparison on a schedule.

Here is the whole loop:

```mermaid
flowchart TD
    A[Training data CSV in S3] -->|suggest_baseline job<br/>runs once| B[constraints.json<br/>statistics.json]
    C[Endpoint] -->|data capture<br/>continuous| D[Captured requests<br/>and responses in S3]
    B --> E{Monitoring job<br/>hourly schedule}
    D --> E
    E -->|compares capture<br/>to baseline| F[Violation report in S3]
    E -->|emits metrics| G[CloudWatch]
```

Three moving parts:

1. **Baseline** - a one-shot Processing job, `suggest_baseline(...)`, reads
   your training data and writes `constraints.json` (the expected schema and
   value ranges per feature) and `statistics.json` (the reference
   distribution) to S3. This is "what normal looks like".
2. **Data capture** - already on. The endpoint's live traffic in S3. This is
   "what is actually happening".
3. **Monitoring schedule** - `create_monitoring_schedule(...)` sets up an
   hourly job that compares (2) against (1). When a feature drifts beyond the
   constraints, it writes a violation report and emits a CloudWatch metric.

What we are about to build, in this notebook, is parts (1) and (3). Part (2)
is done. The goal of this section is a running hourly monitor on your cohort
endpoint that would catch a feature-distribution shift without anyone
watching.

### Demo - the three SageMaker SDK calls you will use

The SageMaker SDK gives you `DefaultModelMonitor` for data-quality
monitoring. You use it in three steps. Read these now; you will write them
in Lab 2.

**Step 1 - construct the monitor.** It describes the compute for the jobs:

```python
from sagemaker.model_monitor import DefaultModelMonitor

monitor = DefaultModelMonitor(
    role=role,                       # the SageMaker execution role ARN
    instance_count=1,
    instance_type="ml.m5.xlarge",    # 16 GiB - smaller instances OOM here
    volume_size_in_gb=20,
    max_runtime_in_seconds=1800,
    sagemaker_session=sm_session,
)
```

**Step 2 - suggest the baseline.** A one-shot Processing job that reads the
training CSV and writes `constraints.json` + `statistics.json` to S3:

```python
from sagemaker.model_monitor.dataset_format import DatasetFormat

monitor.suggest_baseline(
    baseline_dataset="s3://.../baseline.csv",
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri="s3://.../baseline-results",
    wait=False,    # do NOT block the notebook for 5-8 minutes in class
)
```

`wait=False` is important: the baseline job takes several minutes. We start
it and move on; it finishes in the background.

**Step 3 - create the hourly schedule.** This is the monitor that actually
runs forever:

```python
from sagemaker.model_monitor import CronExpressionGenerator

monitor.create_monitoring_schedule(
    monitor_schedule_name="fraud-classifier-hourly",
    endpoint_input=ENDPOINT_NAME,
    output_s3_uri="s3://.../schedule-results",
    statistics=monitor.baseline_statistics(),
    constraints=monitor.suggested_constraints(),
    schedule_cron_expression=CronExpressionGenerator.hourly(),
    enable_cloudwatch_metrics=True,
)
```

That is the entire pattern. In Lab 2 you write these three calls. You will
NOT wait for the hourly job to fire in class - you confirm the schedule
exists in the SageMaker console.

### Lab 2 - Baseline + scheduled monitor (15 min)

Now you write the three SDK calls from the demo above. The lab is mostly
configuration; the point is to wire the three moving parts together.

Your task, in the starter cell below:

1. Construct a `DefaultModelMonitor` (use `instance_type="ml.m5.xlarge"` -
   smaller instances OOM on the baseline job).
2. Call `suggest_baseline(...)` with `wait=False` so the notebook does not
   block for several minutes.
3. Call `create_monitoring_schedule(...)` with an hourly cron, passing
   `monitor_schedule_name=SCHEDULE_NAME`. The starter cell already builds
   `SCHEDULE_NAME` and the S3 output paths PER STUDENT (keyed on your
   student number) so 60 students never collide on one schedule name or
   overwrite each other's results in S3.

You will not wait for the scheduled run to complete in class. You start it
and confirm the schedule shows up under SageMaker -> Monitoring jobs.

### Stretch

Once the baseline job finishes, read the generated `constraints.json` from
`baseline_results_s3` and print the inferred type and completeness for
`amount`.

### Homework extension

Trigger the monitoring job by sending 1000 invocations with deliberately
drifted `amount` values, then read the violation report from S3.

In [ ]:
# Lab 2 SOLUTION. Use SageMaker SDK to baseline and schedule.

from sagemaker import Session
from sagemaker.model_monitor import DefaultModelMonitor, CronExpressionGenerator
from sagemaker.model_monitor.dataset_format import DatasetFormat

sm_session = Session()
role = dbutils.secrets.get(scope="aws-course-shared", key="sagemaker-execution-role-arn")

# The baseline INPUT (training data) is shared and read-only - fine to share.
baseline_input_s3 = f"s3://{bucket}/fraud-classifier/training/baseline.csv"

# The baseline RESULTS and the schedule OUTPUT are per-student work products.
# Key them on the student number so 60 students do not clobber each other in
# S3 or collide on the schedule name. _num was derived in the setup cell.
baseline_results_s3 = f"s3://{bucket}/fraud-classifier/monitoring/student-{_num}/baseline-results"
schedule_output_s3 = f"s3://{bucket}/fraud-classifier/monitoring/student-{_num}/schedule-results"
SCHEDULE_NAME = f"fraud-classifier-hourly-student-{_num}"

# Step 1 - construct the monitor. ml.m5.xlarge (16 GiB) - smaller instances
# OOM on the baseline Processing job.
monitor = DefaultModelMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20,
    max_runtime_in_seconds=1800,
    sagemaker_session=sm_session,
)

# Step 2 - suggest the baseline. wait=False so the notebook does not block
# for the several minutes the Processing job takes.
monitor.suggest_baseline(
    baseline_dataset=baseline_input_s3,
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri=baseline_results_s3,
    wait=False,
)

# Idempotency: drop an existing schedule of the same name so a re-run does
# not hit ResourceInUse.
import time
sm_client = sm_session.boto_session.client("sagemaker")
try:
    sm_client.describe_monitoring_schedule(MonitoringScheduleName=SCHEDULE_NAME)
    print(f"Found existing schedule {SCHEDULE_NAME}, deleting before recreate.")
    sm_client.delete_monitoring_schedule(MonitoringScheduleName=SCHEDULE_NAME)
    for _ in range(12):
        time.sleep(5)
        try:
            sm_client.describe_monitoring_schedule(MonitoringScheduleName=SCHEDULE_NAME)
        except sm_client.exceptions.ClientError:
            break
except sm_client.exceptions.ClientError:
    pass

# Step 3 - create the hourly monitoring schedule on this cohort's endpoint.
monitor.create_monitoring_schedule(
    monitor_schedule_name=SCHEDULE_NAME,
    endpoint_input=ENDPOINT_NAME,
    output_s3_uri=schedule_output_s3,
    statistics=monitor.baseline_statistics(),
    constraints=monitor.suggested_constraints(),
    schedule_cron_expression=CronExpressionGenerator.hourly(),
    enable_cloudwatch_metrics=True,
)

print("Schedule:", SCHEDULE_NAME, "- check SageMaker console -> Monitoring jobs.")


## Part 3 - CloudWatch alarm on endpoint latency

Model Monitor fires on data quality. Langfuse fires on agent behavior. Neither pages you at 3am when the endpoint just becomes slow.

SageMaker endpoints emit metrics to CloudWatch in the `aws/sagemaker/Endpoints` namespace: `ModelLatency`, `Invocations`, `Invocation4XXErrors`, `Invocation5XXErrors`. We will create one alarm: if `ModelLatency` p95 exceeds 1000 ms for 5 minutes, send a message to the SNS topic the instructor pre-created. SNS fans out to email, Slack, PagerDuty, whatever.

Watch the unit: `ModelLatency` is reported in microseconds, so a 1000 ms threshold is `1000000` in the alarm config.

This part is a demo only. You watch the cell run and the alarm appear in the CloudWatch console.

In [ ]:
# Create a CloudWatch alarm wired to an SNS topic the instructor provisioned.

sns_topic_arn = dbutils.secrets.get(scope="aws-course-shared", key="sns-alerts-topic-arn")

# Per-student alarm name so 60 students do not overwrite one shared alarm.
alarm_name = f"fraud-classifier-high-latency-student-{_num}"

cloudwatch.put_metric_alarm(
    AlarmName=alarm_name,
    AlarmDescription="Fires when p95 ModelLatency exceeds 1000 ms for 5 minutes",
    MetricName="ModelLatency",
    Namespace="AWS/SageMaker",
    # ModelLatency is reported in MICROSECONDS. 1000 ms = 1000000 us.
    ExtendedStatistic="p95",
    Dimensions=[
        {"Name": "EndpointName", "Value": ENDPOINT_NAME},
        {"Name": "VariantName", "Value": "AllTraffic"},
    ],
    Period=60,
    EvaluationPeriods=5,
    Threshold=1000000.0,
    ComparisonOperator="GreaterThanThreshold",
    TreatMissingData="notBreaching",
    AlarmActions=[sns_topic_arn],
)

print("Alarm created:", alarm_name)
print("Open CloudWatch -> Alarms to see it.")
print("Subscribe yourself to", sns_topic_arn, "to receive the page.")

## Part 4 - LiteLLM as a unified interface (brief)

Strands gives you agents. Langfuse gives you traces. But sometimes you do not need an agent; you just need to call an LLM and log the call. LiteLLM is the "boring" version of this pattern: a single Python function `litellm.completion(...)` that speaks to any provider, with built-in callbacks for Langfuse and others.

You will not build with LiteLLM today. You will see it work in one cell. The point is the interface: same call, any provider, free Langfuse logging.

In [ ]:
# Single demo of LiteLLM auto-logging to Langfuse. No agent involved.

import litellm

# Tell LiteLLM to log everything to Langfuse. The Langfuse env vars
# from Part 1 are still set, so it picks them up automatically.
litellm.success_callback = ["langfuse"]
litellm.failure_callback = ["langfuse"]

resp = litellm.completion(
    model="bedrock/converse/us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    messages=[
        {"role": "user", "content": "In one sentence, why is online monitoring required for ML systems?"}
    ],
    max_tokens=80,
    temperature=0,
)
print(resp.choices[0].message.content)
print()
print("This call appears in Langfuse as a Generation (not a Trace), tagged with the model name.")

### Think about it

You now have three production signals on the same fraud system:

1. Langfuse traces (per-decision, qualitative)
2. SageMaker Model Monitor (feature distribution, statistical)
3. CloudWatch alarms (operational, time-series)

Which of these would have caught the following, in order of speed?
- The endpoint instance ran out of memory at 3:04 am
- A new fraud pattern using sub-dollar amounts started yesterday
- The supervisor started calling `policy_retriever_v2` 14 times per decision because of a prompt regression you shipped Friday
- The training-serving skew on `merchant_category` has been growing for three weeks

There is no single right ordering. The point is that each signal has a job, and none of them substitutes for the others. A fourth signal, upstream drift detection on the data lake itself, is the subject of Week 21.

## A note on CI/CD

We did not build a GitHub Actions workflow in this notebook. In a Databricks-centric MLOps shop you usually combine two pieces:

1. A GitHub Actions workflow that, on push to `main`, runs unit tests, packages the training code, and submits a Databricks Job that runs the Week 19 SageMaker training script.
2. A scheduled drift check that decides when to retrain and triggers (1) if drift is detected.

The "CI" half is testing your training code before it can run. The "CD" half is the drift check that decides when to run it. You have the "CI" ingredient already; the scheduled drift check is what Week 21 builds with Airflow.

## Wrap-up

You took a deployed Strands-based fraud system and put three production guardrails on it:

- Langfuse + OTEL traces on `week19_supervisor` with five lines of setup
- SageMaker data capture and a Model Monitor schedule on the cohort endpoint
- A CloudWatch alarm on endpoint latency wired to SNS
- A glimpse of LiteLLM as the simpler unified-interface pattern

## Homework

1. Finish the homework extension on Lab 1 (24h replay + p95 dashboard).
2. Read the SageMaker Model Monitor docs for the four built-in monitor types: Data Quality, Model Quality, Bias Drift, Feature Attribution Drift. Pick one of the latter three and write a 200-word note on when you would add it to this pipeline.
3. Preview Week 21: think about which of this week's checks belong on a schedule rather than run by hand, and what should happen automatically when one of them fires.

Next week (Week 21) we move to orchestration with Airflow / MWAA. Upstream drift detection on the data lake, the natural fourth signal, becomes a scheduled DAG there.